# Capítulo 1 · Tu primera llamada a Claude

Curso **Ingeniería de Prompts con Claude** · caso **Andina Salud**, una aseguradora de salud.

En este notebook vas a:

1. Guardar tu API key sin escribirla en el código.
2. Llamar a Claude **sin SDK**, armando la petición web a mano.
3. Llamar a Claude **con el SDK** y leer el objeto `Message`.
4. Probar un **prompt zero-shot** vago y uno preciso, y medir la diferencia.
5. Ver el **sesgo** del zero-shot: lo que Claude rellena cuando el prompt no le da el dato.
6. Convertir el prompt en una pequeña **aplicación** que decide según el objeto `Message`.

Ejecuta las celdas **en orden**, de arriba hacia abajo, con ▶ o con `Shift + Enter`.

## 1 · Instalar el SDK y cargar tu API key

**En Google Colab:** abre el panel de la llave 🔑 (Secretos) a la izquierda, agrega un secreto llamado
`ANTHROPIC_API_KEY` con tu clave y activa el acceso del notebook.

**En tu computador:** crea un archivo `.env` en esta carpeta con la línea
`ANTHROPIC_API_KEY=tu-clave`, y verifica que `.env` esté en tu `.gitignore`.

La celda siguiente funciona en los dos casos.

In [ ]:
%pip install -q anthropic python-dotenv requests

In [ ]:
import os

try:
    from google.colab import userdata          # en Colab: Secretos
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("Clave cargada desde los Secretos de Colab.")
except ImportError:
    from dotenv import load_dotenv             # en tu computador: archivo .env
    load_dotenv()
    print("Clave cargada desde el archivo .env.")

assert os.environ.get("ANTHROPIC_API_KEY"), "No encontré ANTHROPIC_API_KEY. Revisa el paso anterior."

## 2 · Sin SDK: la petición web armada a mano

Así se ve una llamada a Claude sin ninguna ayuda. Tienes que escribir la dirección, las tres cabeceras y el
cuerpo en JSON, y después desarmar la respuesta tú mismo. Si la API falla por un problema pasajero, este código
no reintenta.

In [ ]:
import requests

LLAMADA = ("Buenos días, llamo porque me rechazaron el reembolso de una resonancia magnética. "
           "Me dicen que falta la orden médica, pero yo la adjunté.")

respuesta = requests.post(
    "https://api.anthropic.com/v1/messages",
    headers={
        "x-api-key": os.environ["ANTHROPIC_API_KEY"],
        "anthropic-version": "2023-06-01",
        "content-type": "application/json",
    },
    json={
        "model": "claude-haiku-4-5",
        "max_tokens": 200,
        "messages": [{"role": "user", "content": f"Resume en una frase el motivo de esta llamada: {LLAMADA}"}],
    },
    timeout=60,
)

datos = respuesta.json()
print(datos)                          # la respuesta completa, en JSON
print()
if respuesta.status_code == 200:
    print(datos["content"][0]["text"])    # solo el texto
else:
    print(f"Error {respuesta.status_code}: {datos['error']['message']}")

## 3 · Con el SDK: la misma llamada

El SDK pone la dirección y las cabeceras, lee la clave del entorno, reintenta los errores pasajeros y te
devuelve un objeto `Message`.

In [ ]:
import os

try:
    import anthropic
except ModuleNotFoundError:                     # no se ejecutó la sección 1
    %pip install -q anthropic python-dotenv
    import anthropic

if not os.environ.get("ANTHROPIC_API_KEY"):     # no se cargó la clave
    try:
        from google.colab import userdata      # en Colab: Secretos
        os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    except ImportError:
        from dotenv import load_dotenv         # en tu computador: .env
        load_dotenv()

if "LLAMADA" not in globals():                  # no se ejecutó la sección 2
    LLAMADA = ("Buenos días, llamo porque me rechazaron el reembolso de una resonancia magnética. "
               "Me dicen que falta la orden médica, pero yo la adjunté.")

client = anthropic.Anthropic()        # lee ANTHROPIC_API_KEY del entorno

mensaje = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=200,
    messages=[{"role": "user", "content": f"Resume en una frase el motivo de esta llamada: {LLAMADA}"}],
)

print(mensaje)                        # el objeto Message entero

### Leer la respuesta en un solo lugar

El texto no está en `mensaje.text`: está dentro de la lista `content`. Esta función lo junta, y el resto del
programa solo usa ella.

In [ ]:
def leer_texto(mensaje):
    """Junta el texto de todos los bloques de tipo text."""
    return "".join(bloque.text for bloque in mensaje.content if bloque.type == "text")


print("Texto:      ", leer_texto(mensaje))
print("stop_reason:", mensaje.stop_reason)
print("tokens:     ", mensaje.usage.input_tokens, "de entrada +", mensaje.usage.output_tokens, "de salida")
print("id:         ", mensaje.id)

## 4 · Zero-shot vago y zero-shot preciso

Los dos prompts son **zero-shot**: dan la tarea sin ningún ejemplo de respuesta. Cada uno se ejecuta tres
veces. Mira cuánto cambia el largo.

In [ ]:
VAGO = "Resume esta llamada:"
PRECISO = "Resume en una frase de máximo 20 palabras el motivo de esta llamada, sin saludos:"

for nombre, instruccion in [("vago", VAGO), ("preciso", PRECISO)]:
    for intento in range(3):
        m = client.messages.create(
            model="claude-haiku-4-5",
            max_tokens=300,
            messages=[{"role": "user", "content": f"{instruccion}\n\n{LLAMADA}"}],
        )
        texto = leer_texto(m)
        print(f"{nombre} #{intento + 1}: {len(texto.split())} palabras -> {texto[:90]}")

## 5 · El sesgo del zero-shot

Sin el dato, Claude responde con lo más común que aprendió en su entrenamiento, no con lo que dice Andina Salud.
Con el dato y el permiso para no saber, responde con el manual.

In [ ]:
MANUAL = ("El reembolso se abona en un plazo de 10 días hábiles desde la presentación "
          "completa de los documentos.")


def preguntar(pregunta, con_manual):
    if con_manual:
        prompt = ("Responde solo con lo que dice el manual. Si no está, responde: No figura en el manual.\n"
                  f"<manual>{MANUAL}</manual>\n{pregunta}")
    else:
        prompt = pregunta
    m = client.messages.create(model="claude-haiku-4-5", max_tokens=200,
                               messages=[{"role": "user", "content": prompt}])
    return leer_texto(m)


for pregunta in ["¿Cuánto tarda el reembolso de una consulta médica?", "¿Cubre la ortodoncia?"]:
    print("PREGUNTA:", pregunta)
    print("  sin manual:", preguntar(pregunta, False))
    print("  con manual:", preguntar(pregunta, True))
    print()

## 6 · Del prompt a una aplicación

La central de Andina Salud llamaría a esta función al cerrar cada llamada. El prompt sigue siendo zero-shot; lo
nuevo es que la aplicación **decide qué hacer** con los campos del objeto `Message`.

In [ ]:
INSTRUCCION = "Resume en una frase de máximo 25 palabras el motivo de esta llamada, sin saludos:"


def resumir_llamada(transcripcion, max_tokens=150):
    mensaje = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": f"{INSTRUCCION}\n\n{transcripcion}"}],
    )
    resultado = {
        "resumen": leer_texto(mensaje),
        "completo": mensaje.stop_reason == "end_turn",
        "tokens_entrada": mensaje.usage.input_tokens,
        "tokens_salida": mensaje.usage.output_tokens,
        "id": mensaje.id,
    }
    if mensaje.stop_reason == "max_tokens":
        resultado["alerta"] = "El resumen se cortó: revisarlo a mano o subir max_tokens."
    return resultado


try:
    for tope in (150, 5):                     # con 5 tokens el resumen se corta a propósito
        r = resumir_llamada(LLAMADA, max_tokens=tope)
        if r["completo"]:
            print(f"max_tokens={tope} -> Resumen:", r["resumen"])
        else:
            print(f"max_tokens={tope} -> Revisar:", r["alerta"], "| texto cortado:", r["resumen"])
        print("   tokens:", r["tokens_entrada"], "+", r["tokens_salida"], "· id:", r["id"])
except anthropic.AuthenticationError:
    print("La API key no es válida: revisa el paso 1.")
except anthropic.RateLimitError:
    print("La API está saturada después de los reintentos: espera un momento y vuelve a ejecutar.")

## Ejercicios

1. **Tu transcripción.** Cambia `LLAMADA` por una llamada inventada por ti y vuelve a ejecutar las secciones 3 a 6.
2. **Mide el zero-shot.** Anota en una tabla el largo de las seis respuestas de la sección 4.
3. **Donde el zero-shot no alcanza.** Pide «Clasifica esta llamada en su categoría de reclamo» tres veces.
   ¿Qué categorías inventa Claude?
4. **Encuentra el sesgo.** En la sección 5, pregunta por algo que el manual no dice. ¿En qué caso inventa?
5. **Sin clave.** Borra el secreto (o el `.env`) y ejecuta la sección 3. Anota el error: es el 401.